In [1]:
import sys, importlib.util, dataclasses
from pathlib import Path
from datetime import date
import pandas as pd

for _c in [Path.cwd()] + list(Path.cwd().parents):
    if (_c / "gbp").is_dir() and (_c / "tests").is_dir():
        sys.path.insert(0, str(_c)); break

_spec = importlib.util.spec_from_file_location("fixtures", _c / "tests/unit/build/fixtures.py")
_mod = importlib.util.module_from_spec(_spec); _spec.loader.exec_module(_mod)
minimal_raw_model = _mod.minimal_raw_model

from gbp.core.enums import FacilityRole, FacilityType, ModalType, OperationType, PeriodType, AttributeKind, ResourceStatus
from gbp.core.roles import DEFAULT_ROLES, derive_roles
from gbp.core.model import RawModelData, ResolvedModelData
from gbp.build.pipeline import build_model
from gbp.build.validation import validate_raw_model
from gbp.build.time_resolution import resolve_all_time_varying, resolve_registry_attributes
from gbp.build.lead_time import resolve_lead_times
from gbp.build.transformation import resolve_transformations
from gbp.build.fleet_capacity import compute_fleet_capacity
from gbp.build.spine import assemble_spines
from gbp.consumers.simulator.state import init_state


# --- L2 enums: domain-agnostic ---

list(FacilityRole)    # source, sink, storage, transshipment
list(ModalType)       # road, rail, sea, pipeline, air, digital
list(PeriodType)      # day, week, month
list(AttributeKind)   # cost, revenue, rate, capacity, additional
list(ResourceStatus)  # available, in_transit, busy, maintenance

# --- L3 enums: bike-sharing specific ---

list(FacilityType)    # station, depot, maintenance_hub
list(OperationType)   # receiving, storage, dispatch, handling, repair


# --- Role derivation: type + operations -> roles ---

DEFAULT_ROLES  # station -> {source,sink,storage}, depot -> {storage,transshipment}, hub -> {transshipment,storage}

derive_roles("station", {"receiving", "storage", "dispatch"})  # {source, sink, storage, transshipment}
derive_roles("depot",   {"receiving", "storage", "dispatch"})  # {storage, transshipment}
derive_roles("depot",   {"receiving", "dispatch"})             # no storage op -> STORAGE role discarded
derive_roles("station", set(), {FacilityRole.SINK})            # manual override ignores defaults


# --- RawModelData: 9 required + optional tables ---

raw = minimal_raw_model(with_demand=True)

RawModelData._REQUIRED   # 9 required: facilities, commodity_categories, resource_categories, planning_horizon, planning_horizon_segments, periods, facility_roles, facility_operations, edge_rules
RawModelData._GROUPS     # logical groups: entity, temporal, behavior, edge, flow_data, transformation, resource, hierarchy, scenario

# entity group
raw.facilities            # facility_id, facility_type, name -- graph nodes (d1=depot, s1/s2=stations)
raw.commodity_categories  # commodity_category_id, name, unit -- what flows (working_bike)
raw.resource_categories   # resource_category_id, name, base_capacity -- what moves it (rebalancing_truck)

# temporal group: PlanningHorizon -> Segments -> Periods
raw.planning_horizon             # h1: 2025-01-01 .. 2025-01-04
raw.planning_horizon_segments    # 1 segment, DAY granularity
raw.periods                      # p0, p1, p2 -- one per day, period_index is global & continuous for carry-over

# behavior group
raw.facility_roles       # d1: {storage,transshipment}, s1: {sink,storage,source}, s2: {sink,storage}
raw.facility_operations  # all facilities: receiving, storage, dispatch -- all enabled
raw.edge_rules           # depot->station for working_bike on road

# edge group -- PK = (source_id, target_id, modal_type)
raw.edges                # d1->s1 (1km, 24h), d1->s2 (2km, 48h) -- both road
raw.edge_commodities     # which commodities allowed on each edge (working_bike on both)

# flow group -- time-varying, keyed by 'date' in raw
raw.demand               # s1: 1 bike on Jan 1, 2 on Jan 2; s2: 3 on Jan 1
raw.supply               # None
raw.inventory_initial    # None
raw.inventory_in_transit # None

# resource group
raw.resource_fleet                    # 3 trucks at d1
raw.resource_commodity_compatibility  # truck <-> working_bike
raw.resource_modal_compatibility      # truck <-> road

# parametric attributes (AttributeRegistry, not a DataFrame)
raw.attributes  # empty in this fixture; real loaders register operation_capacity, operation_cost, etc.


# --- Validation + navigation ---

raw.validate()            # checks: 9 required present + columns match Pydantic row schemas
raw.entity_tables         # {facilities, commodity_categories, resource_categories}
raw.temporal_tables       # {planning_horizon, planning_horizon_segments, periods}
raw.behavior_tables       # {facility_roles, facility_operations, edge_rules}
raw.edge_tables           # {edges, edge_commodities}
raw.flow_tables           # {demand}
raw.resource_tables       # {resource_commodity_compatibility, resource_modal_compatibility, resource_fleet}
raw.parameter_tables      # {} -- from AttributeRegistry
raw.populated_tables      # all non-None DataFrames + registry attrs


# --- build_model() step by step ---

# 1. validate
validate_raw_model(raw).raise_if_invalid()

# 2. resolve time-varying tables: date -> period_id with aggregation
periods = raw.periods.copy()
resolved_time = resolve_all_time_varying(raw, periods)  # {'demand': DataFrame with period_id instead of date}
resolved_time["demand"]  # facility_id, commodity_category, period_id, quantity (aggregated by sum)

# 3. resolve parametric attributes (date -> period_id in registry)
resolved_attrs = resolve_registry_attributes(raw.attributes, periods)

# 4. ensure edges exist (use raw.edges, or build from edge_rules)
edges_df = raw.edges
ec_df = raw.edge_commodities

# 5. resolve lead times: hours -> periods per edge x period
elt = resolve_lead_times(edges_df, periods)
elt  # d1->s1: 24h = 1 period; d1->s2: 48h = 2 periods (DAY granularity)

# 6. resolve transformations (N:M commodity conversion -- none here)
transformation_resolved = resolve_transformations(
    raw.facilities, raw.transformations, raw.transformation_inputs, raw.transformation_outputs,
)

# 7. compute fleet capacity: resource_fleet x resource_categories -> fleet_capacity
fleet_capacity = compute_fleet_capacity(raw.resource_fleet, raw.resource_categories, raw.resources)
fleet_capacity  # facility_id=d1, resource_category=rebalancing_truck, count=3, total_capacity=60

# 8. assemble ResolvedModelData
resolved = ResolvedModelData.from_raw(
    raw, periods=periods, resolved_time=resolved_time, resolved_attrs=resolved_attrs,
    edges=edges_df, edge_commodities=ec_df, edge_lead_time_resolved=elt,
    transformation_resolved=transformation_resolved, fleet_capacity=fleet_capacity,
)

# 9. assemble spines (attribute DataFrames grouped by entity type for vectorized lookups)
spines = assemble_spines(resolved)
resolved.facility_spines = spines["facility"] or None
resolved.edge_spines = spines["edge"] or None
resolved.resource_spines = spines["resource"] or None


# --- What changed: Raw -> Resolved ---

raw.demand       # date-keyed: facility_id, commodity_category, date, quantity
resolved.demand  # period_id-keyed: facility_id, commodity_category, period_id, quantity (aggregated)

resolved.edge_lead_time_resolved  # new: edge x period -> lead_time_periods (integer)
resolved.fleet_capacity           # new: facility x resource_category -> count, total_capacity
resolved.resources                # new: L3 instances expanded from fleet (truck_0, truck_1, truck_2)

resolved.generated_tables  # {edge_lead_time_resolved, fleet_capacity}
resolved.spine_tables      # {facility_spines, edge_spines, resource_spines}


# --- Parametric costs via AttributeRegistry ---

raw_costs = minimal_raw_model(with_demand=True, with_costs=True)
raw_costs.attributes  # 4 registered: operation_cost, transport_cost, resource_fixed_cost, resource_maintenance_cost

# operation_cost: facility x operation x commodity x date -> cost_per_unit
raw_costs.attributes.get("operation_cost").data  # d1 dispatch 0.5 usd, s1/s2 receiving 1.0-1.5 usd

# transport_cost: edge x resource x date -> cost_per_unit
raw_costs.attributes.get("transport_cost").data  # d1->s1 2.0 usd, d1->s2 3.5 usd per bike

# resource costs (EAV): resource_category x date -> value (filtered by attribute_name)
raw_costs.attributes.get("resource_fixed_cost").data       # truck: 10.0 usd/period
raw_costs.attributes.get("resource_maintenance_cost").data  # truck: 2.5 usd/period

# costs flow through build_model -> resolved attributes -> spines
resolved_costs = build_model(raw_costs)
resolved_costs.attributes  # same 4 attrs, dates resolved to period_ids
resolved_costs.parameter_tables  # dict of all parametric attribute DataFrames


# --- Same ResolvedModelData -> all consumers ---

raw_full = dataclasses.replace(raw_costs, inventory_initial=pd.DataFrame({
    "facility_id": ["d1", "s1", "s2"],
    "commodity_category": ["working_bike"] * 3,
    "quantity": [50.0, 10.0, 5.0],
    "quantity_unit": ["bike"] * 3,
}))
resolved_full = build_model(raw_full)

state = init_state(resolved_full)
state.period_id         # p0
state.period_index      # 0
state.inventory         # facility x commodity -> quantity (d1=50, s1=10, s2=5)
state.resources         # 3 truck instances, status=available, at d1
state.in_transit        # empty -- no pre-existing shipments

,shipment_id,source_id,target_id,commodity_category,quantity,resource_id,departure_period,arrival_period


In [24]:
resolved_costs.spine_tables['facility_spines']['group_0']

,facility_id,facility_type,name,operation_type,commodity_category,period_id,operation_cost
0,d1,depot,Depot,dispatch,working_bike,p0,0.5
1,s1,station,Station 1,receiving,working_bike,p0,1.0
2,s1,station,Station 1,receiving,working_bike,p1,1.0
3,s2,station,Station 2,receiving,working_bike,p0,1.5


In [25]:
resolved_costs.spine_tables['edge_spines']['group_0']

,source_id,target_id,modal_type,distance,distance_unit,lead_time_hours,reliability,edge_distance_x,edge_distance_y,edge_lead_time_hours_x,edge_lead_time_hours_y,edge_reliability_x,edge_reliability_y,resource_category,period_id,transport_cost
0,d1,s1,road,1.0,km,24.0,0.99,1.0,1.0,24.0,24.0,0.99,0.99,rebalancing_truck,p0,2.0
1,d1,s2,road,2.0,km,48.0,0.99,2.0,2.0,48.0,48.0,0.99,0.99,rebalancing_truck,p0,3.5


In [26]:
resolved_costs.spine_tables['resource_spines']['group_0']

,resource_category,name,base_capacity,capacity_unit,resource_base_capacity_x,resource_base_capacity_y,period_id,resource_fixed_cost,resource_maintenance_cost
0,rebalancing_truck,Truck,20.0,bike,20.0,20.0,p0,10.0,2.5
